# 00 Environment and data audit

**Question.** Is this machine ready for study v2 (environment, data, weights, split, provenance)?

**Type.** Inspection only. Nothing is trained and no model is run. **Prerequisites.** `uv sync --extra dev`.
Outputs are not stored in this notebook; run it to see them.

In [ ]:
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import json

import pandas as pd
from IPython.display import display

from ct_iqa.data.splits import load_manifest
from ct_iqa.experiment.metadata import environment_info, git_info
from ct_iqa.models.weights import sha256_file

checks = {}   # name -> bool, summarized at the end


## Environment and Git state

In [ ]:
info = {**environment_info("auto"), **{f"git_{k}": v for k, v in git_info(REPO).items()}}
display(pd.DataFrame({"value": info}))


## Labels and frozen split
Checks label counts, score range and that the manifest partitions are disjoint and cover the labeled images.

In [ ]:
labels = json.loads(Path("data/labels/ldct_iqac/train.json").read_text())
test_labels = json.loads(Path("data/labels/ldct_iqac/test.json").read_text())   # counted only; scores are not analysed
manifest = load_manifest("experiments/study_v2/protocol/split_manifest.json", labels)   # raises if overlapping or incomplete
checks["train labels = 1000"] = len(labels) == 1000
checks["test labels = 300"] = len(test_labels) == 300
checks["train scores within [0, 4]"] = 0 <= min(labels.values()) and max(labels.values()) <= 4
checks["split partitions 800/100/100"] = [len(manifest[k]) for k in ("train", "validation", "calibration")] == [800, 100, 100]
print("manifest sha256:", manifest["manifest_sha256"])
display(pd.DataFrame(manifest["label_statistics_by_partition"]).T)


## Weights and historical checkpoints
Verifies the converted RadImageNet weights against the recorded SHA-256 and reports which historical checkpoints exist.

In [ ]:
weights_dir = Path("weights/pretrained/radimagenet/resnet50")
rows = []
for line in (weights_dir / "CHECKSUMS.sha256").read_text().splitlines():
    digest, name = line.split()[0], line.split()[-1].lstrip("*")
    path = weights_dir / name
    ok = path.is_file() and sha256_file(path) == digest
    rows.append({"file": name, "present": path.is_file(), "sha256 matches": ok})
    checks[f"weights {name}"] = ok
display(pd.DataFrame(rows))
display(pd.DataFrame({"historical checkpoint": [str(p) for p in sorted(Path("experiments").glob("00*/**/checkpoint/*.pt"))]}))


## Raw image availability
Counts files only; images are not read.

In [ ]:
counts = {d: len(list(Path(d).glob("*.tif*"))) if Path(d).is_dir() else None
          for d in ("data/raw/ldct_iqac/train/image", "data/raw/ldct_iqac/test/images")}
display(pd.DataFrame({"image files": counts}))
checks["train images = 1000"] = counts["data/raw/ldct_iqac/train/image"] == 1000
checks["test images = 300"] = counts["data/raw/ldct_iqac/test/images"] == 300


## Summary

In [ ]:
summary = pd.DataFrame({"passed": checks})
display(summary)
print("ALL CHECKS PASSED" if summary["passed"].all() else "SOME CHECKS FAILED - fix before running experiments")
